# Surgical Annotation Pipeline: Fine-tuned YOLOv8 + SAM 2

This notebook runs the complete workflow:
1. Download CholecSeg8k dataset (2000 images)
2. Convert to YOLO training format
3. Fine-tune YOLOv8 on surgical instruments and organs
4. Evaluate against baseline (Grounding DINO)
5. Show side-by-side comparison metrics

**Expected time: 2-3 hours total on T4 GPU**
- Dataset download: ~15 min (3GB zip)
- YOLO training: 60-90 min (2000 images, 50 epochs)
- Comparison evaluation: ~15 min

**Prerequisites:**
1. GPU enabled (T4)
2. Project uploaded to `/content/surgical-video-annotator-final`
3. All previous dependencies installed

## Step 1: Verify environment

In [ ]:
!nvidia-smi

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')

## Step 2: Install ultralytics (YOLO)

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
os.chdir('/content/surgical-video-annotator-final')
!ls src/ scripts/

## Step 3: Download CholecSeg8k (2000 samples)

First run downloads the full 3GB zip file (10-15 min). Cached afterwards.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.dataset_loader import CholecSeg8kLoader

loader = CholecSeg8kLoader(cache_dir='cholecseg8k_data')
samples = loader.download_subset(n=2000)
print(f'\nReady: {len(samples)} samples')

## Step 4: Convert to YOLO training format

Creates the folder structure YOLO expects with train/val/test splits.

In [ ]:
yolo_config = loader.export_yolo_dataset(
    output_dir='yolo_dataset',
    train_split=0.8,
    val_split=0.1,
    include_anatomy=True,
)

print(f'\nDataset ready:')
print(f'  Classes: {yolo_config["num_classes"]}')
print(f'  Class names: {yolo_config["class_names"]}')
print(f'  Splits: {yolo_config["splits"]}')

## Step 5: Train YOLOv8

This is the main event. Fine-tunes YOLOv8 nano (fastest variant) on our surgical data.

**Time: 60-90 minutes on T4 GPU.** Get coffee.

You'll see live training metrics printed. Look for:
- Loss going down
- mAP@0.5 going up (target: >0.4 for good results)
- Early stopping if it plateaus

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 (starts from COCO weights)
model = YOLO('yolov8n.pt')

# Train on surgical data
results = model.train(
    data=yolo_config['data_yaml'],
    epochs=50,
    imgsz=640,
    batch=16,
    project='yolo_models',
    name='surgical_yolov8',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    patience=15,
    save=True,
    verbose=True,
)

print('\n\nTraining complete!')
print(f'Best weights: yolo_models/surgical_yolov8/weights/best.pt')

## Step 6: Evaluate on test set

In [ ]:
best_weights = 'yolo_models/surgical_yolov8/weights/best.pt'

trained_model = YOLO(best_weights)
test_results = trained_model.val(
    data=yolo_config['data_yaml'],
    split='test',
    imgsz=640,
    batch=16,
)

print(f'\n{"="*70}')
print('YOLO TEST SET RESULTS')
print(f'{"="*70}')
print(f'mAP@0.5:      {test_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {test_results.box.map:.4f}')
print(f'Precision:    {test_results.box.mp:.4f}')
print(f'Recall:       {test_results.box.mr:.4f}')

print(f'\nPer-class mAP@0.5:')
if hasattr(test_results.box, 'ap50') and hasattr(test_results, 'names'):
    for i, ap in enumerate(test_results.box.ap50):
        name = test_results.names.get(i, f'class_{i}')
        print(f'  {name}: {ap:.4f}')

## Step 7: Compare against baseline (Grounding DINO)

This is the money shot. Side-by-side metrics showing your improvement.

In [ ]:
# Update config to point to trained YOLO
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['yolo_detection']['weights_path'] = best_weights
config['detector_type'] = 'yolo'

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated to use trained YOLO weights')

In [ ]:
# Run comparison
from scripts.compare_detectors import compare_detectors

comparison = compare_detectors(
    yolo_weights=best_weights,
    n_test_samples=200,
    output_dir='comparison_results',
)

## Step 8: Visualize results

See side-by-side: Grounding DINO predictions vs YOLO predictions vs ground truth.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import json

with open('comparison_results/test_ground_truth.json', 'r') as f:
    gt = json.load(f)
with open('comparison_results/baseline/predictions_coco.json', 'r') as f:
    baseline_pred = json.load(f)
with open('comparison_results/yolo/predictions_coco.json', 'r') as f:
    yolo_pred = json.load(f)

def build_by_image(coco):
    result = {}
    for a in coco['annotations']:
        result.setdefault(a['image_id'], []).append(a)
    return result

gt_by = build_by_image(gt)
baseline_by = build_by_image(baseline_pred)
yolo_by = build_by_image(yolo_pred)

gt_cats = {c['id']: c['name'] for c in gt['categories']}
baseline_cats = {c['id']: c['name'] for c in baseline_pred['categories']}
yolo_cats = {c['id']: c['name'] for c in yolo_pred['categories']}

def draw_boxes(img, anns, cats, color):
    for a in anns:
        x, y, w, h = [int(v) for v in a['bbox']]
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        label = cats.get(a['category_id'], '?')
        cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

# Show 3 test samples
sample_ids = list(gt_by.keys())[:3]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(20, 6 * len(sample_ids)))

for i, img_id in enumerate(sample_ids):
    img_info = next((im for im in gt['images'] if im['id'] == img_id), None)
    if not img_info:
        continue
    img_path = os.path.join('cholecseg8k_data/images', img_info['file_name'])
    if not os.path.exists(img_path):
        continue
    img = cv2.imread(img_path)
    
    img_gt = draw_boxes(img.copy(), gt_by.get(img_id, []), gt_cats, (0, 255, 0))
    img_base = draw_boxes(img.copy(), baseline_by.get(img_id, []), baseline_cats, (0, 0, 255))
    img_yolo = draw_boxes(img.copy(), yolo_by.get(img_id, []), yolo_cats, (255, 0, 0))
    
    axes[i, 0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'Ground Truth')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(cv2.cvtColor(img_base, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f'Baseline (Grounding DINO)')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(cv2.cvtColor(img_yolo, cv2.COLOR_BGR2RGB))
    axes[i, 2].set_title(f'Fine-tuned YOLOv8')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Step 9: Download everything for your portfolio

In [ ]:
# Zip up the trained model + comparison results + evaluations
!zip -r surgical_annotator_results.zip \
    yolo_models/surgical_yolov8/weights/best.pt \
    yolo_models/surgical_yolov8/results.png \
    comparison_results/ \
    2>&1 | tail -5

from google.colab import files
files.download('surgical_annotator_results.zip')